# Graphaufbau

Dieses Notebook baut einen Knowledge Graphen aus einer CSV Datei für meinen Anwendungsfall

Es führt folgende Schritte aus:
1. Aufbau der Verbindung zu Neo4j
2. Anlegen von Constraints und Indexen
3. Import der Daten aus `data.csv`
4. Erzeugung aller Knoten (`Sensor`, `Measurement`, `Epoch`, `Week`, `Date`, `Hour`, `Minute`)
5. Erzeugung aller Beziehungen

Die Datei `data.csv` muss im `import`-Verzeichnis deiner Neo4j-Installation liegen.

In [ ]:
from neo4j import GraphDatabase

#Verbindungsdaten
uri = "neo4j://neo4j:7687"  
user = "neo4j"
password = "password" 

driver = GraphDatabase.driver(uri, auth=(user, password))
print("Verbindung zu Neo4j hergestellt.")

#Hilfsfunktion zum Ausführen von Cypher-Statements und Konsumieren der Ergebnisse.
def run_cypher(query: str, params: dict | None = None):
    
    if params is None:
        params = {}
    with driver.session() as session:
        result = session.run(query, params)
        summary = result.consume()
    return summary

def print_summary(summary):
    """Kurze Zusammenfassung für Schreib-Queries ausgeben."""
    counters = summary.counters
    print("Nodes created:", counters.nodes_created)
    print("Relationships created:", counters.relationships_created)
    print("Properties set:", counters.properties_set)
    print("Labels added:", counters.labels_added)
    print("Indexes added:", counters.indexes_added)
    print("Constraints added:", counters.constraints_added)

# 1. Alle Daten löschen
print("Lösche alle Nodes & Relationen…")
print_summary(run_cypher("""
MATCH (n)
CALL (n) {
DETACH DELETE n
} IN TRANSACTIONS;
"""))


Verbindung zu Neo4j hergestellt.
Lösche alle Nodes & Relationen…
Nodes created: 0
Relationships created: 0
Properties set: 0
Labels added: 0
Indexes added: 0
Constraints added: 0
Lösche alle Constraints…


ClientError: {code: Neo.ClientError.Procedure.ProcedureNotFound} {message: There is no procedure with the name `db.constraints` registered for this database instance. Please ensure you've spelled the procedure name correctly and that the procedure is properly deployed.}

## 1. Constraints & Indexe anlegen

Diese Constraints stellen sicher, dass wichtige Knoten eindeutig identifizierbar sind
Bei größen Daten können Indizes zusätzlich sinnvoll sein

In [ ]:
constraints = [
    "CREATE CONSTRAINT sensor_moteid_unique IF NOT EXISTS FOR (s:Sensor) REQUIRE s.moteid IS UNIQUE",
    "CREATE CONSTRAINT epoch_id_unique IF NOT EXISTS FOR (e:Epoch) REQUIRE e.epoch_id IS UNIQUE",
    "CREATE CONSTRAINT date_unique IF NOT EXISTS FOR (d:Date) REQUIRE d.date IS UNIQUE",
    "CREATE CONSTRAINT hour_unique IF NOT EXISTS FOR (h:Hour) REQUIRE h.hour IS UNIQUE",
    "CREATE CONSTRAINT minute_unique IF NOT EXISTS FOR (m:Minute) REQUIRE m.minute IS UNIQUE",
    "CREATE CONSTRAINT week_unique IF NOT EXISTS FOR (w:Week) REQUIRE w.week IS UNIQUE"
]

for constraint in constraints:
    summary = run_cypher(constraint)


## 2. CSV-Import 
In diesem Schritt wird die CSV-Datei eingelesen und der Graph angelegt.


    

In [ ]:
import_query = """
CALL {
  LOAD CSV WITH HEADERS FROM 'file:///data.csv' AS row      //Lade Daten

  WITH row,
       row.date AS date,
       row.date + 'T' + row.time AS timestamp,
       toInteger(row.moteid) AS moteid,
       toInteger(row.epoch) AS epoch,
       toFloat(row.temperature) AS temp,
       toFloat(row.humidity) AS hum,
       toFloat(row.light) AS light,
       toFloat(row.voltage) AS volt,
       toInteger(split(row.time, ':')[0]) AS hour,
       toInteger(split(split(row.time, ':')[1], '.')[0]) AS minute,
       date(row.date) AS d,
       date(row.date).week AS week

  MERGE (s:Sensor {mote_id: moteid})                  //Grundstruktur anlegen mit MERGE
  MERGE (e:Epoch {epoch_id: epoch})
  MERGE (w:Week {week: week})
  MERGE (dt:Date {date: date})
  MERGE (w)-[:HAS_DATE]->(dt)
  MERGE (h:Hour {hour: hour})
  MERGE (dt)-[:HAS_HOUR]->(h)
  MERGE (min:Minute {minute: minute})
  MERGE (h)-[:HAS_MINUTE]->(min)

  CREATE (m:Measurement {                               //Mit CREATE Jede Messung anlegen
    temperature: temp,
    humidity: hum,
    light: light,
    voltage: volt,
    time: timestamp
  })

  MERGE (s)-[:GENERATED]->(m)                          //Beziehungen anlegen
  MERGE (m)-[:EPOCH_IN]->(e)
  MERGE (m)-[:AT_WEEK]->(w)
  MERGE (m)-[:AT_DATE]->(dt)
  MERGE (m)-[:AT_HOUR]->(h)
  MERGE (m)-[:AT_MINUTE]->(min)
  MERGE (e)-[:IN_MINUTE]->(min)
} IN TRANSACTIONS OF 5000 ROWS;                       //Aufteilung in Batches falls nötigt
"""

summary = run_cypher(import_query)
print("Import abgeschlossen.")
print_summary(summary)

Import abgeschlossen.
Nodes created: 323558
Relationships created: 1941348
Properties set: 1941348
Labels added: 323558
Indexes added: 0
Constraints added: 0
